In [ ]:
together_API = "API"

LangChain is a Python (and JS) framework for building applications powered by language models (like OpenAI’s GPT, Cohere, etc.)

In [ ]:
!pip install langchain langchain-text-splitters langchain-together langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.3/438.3 kB 9.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.59
    Uninstalling langchain-core-0.3.59:
      Successfully uninstalled langchain-core-0.3.59


In [ ]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.7 MB/s eta 0:00:00


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_together import ChatTogether
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
import os

In [ ]:
llm = ChatTogether(
    together_api_key = "API",
    model = "mistralai/Mistral-7B-Instruct-v0.1",
)

In [ ]:
llm.invoke("hello")

AIMessage(content=' Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 10, 'total_tokens': 20, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'cached_tokens': 0}, 'model_name': 'mistralai/Mistral-7B-Instruct-v0.1', 'system_fingerprint': None, 'id': 'nvj4VZM-3NKUce-9466009adee7f83e', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--9c9c2028-59d2-4e90-845b-c001db06719b-0', usage_metadata={'input_tokens': 10, 'output_tokens': 10, 'total_tokens': 20, 'input_token_details': {}, 'output_token_details': {}})

In [ ]:
from langchain_core.output_parsers import StrOutputParser


You need StrOutputParser() because LangChain LLMs return structured outputs (often message objects or dictionaries), not just raw strings. StrOutputParser extracts and returns the plain text response.

In [ ]:
(llm|StrOutputParser()).invoke("answer in a sentence.which countries were most represented in the olympics 2024?")

' The countries with the most representation in the Olympics 2024 would be determined by the specific games and their respective host countries.'

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://en.wikipedia.org/wiki/2024_Summer_Olympics")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1200,
    chunk_overlap  = 100,
    separators = ["\n\n", "\n", "(?<=\. )", " ", ""]
)

In [ ]:
documents = loader.load()

In [ ]:
docs_split = text_splitter.split_documents(documents)

In [ ]:
embedding_function = SentenceTransformerEmbeddings(model_name="BAAI/bge-small-en-v1.5")

<ipython-input-13-a2f2e6ba7a45>:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = SentenceTransformerEmbeddings(model_name="BAAI/bge-small-en-v1.5")


In [ ]:
!pip install langchain_chroma

The following code creates a Chroma vector store backed by disk persistence, where:

collection_name is your vector DB "table" name.

embedding_function is the model that turns text into vector embeddings.

persist_directory is where the data is stored on disk (./vectordb).

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name = "collection-1",
    embedding_function = embedding_function,
    persist_directory="./vectordb",
)

In [ ]:
retriever = vector_store.as_retriever()

In [ ]:
retriever.add_documents(docs_split)

['f928f0a2-9e1e-4c44-86e5-1d5ade690633',
 'ef61f46b-32e4-454d-a762-f7dc078f4aaa',
 '8ddd47f7-9a75-4faf-aa62-f4382d7b46ed',
 '6f43ffb8-9660-4a66-867e-02630a29bfd6',
 'e3d3c2c3-a2ff-4fb0-b038-646d49adf3ff',
 'a953477e-3cea-42a5-af88-a1e3c8174913',
 'a8527d42-e11f-40ab-b953-5da857dc3f60',
 '1da653b4-5651-464e-9a74-294fccec137a',
 '1d1da396-fc4f-4ba8-b4dc-8c3265fc2aca',
 'db2ccec9-d29a-4161-b897-c0e353c7af55',
 'd5bb022a-1dbf-4de5-9858-f208e59e06f3',
 '07065bff-9035-4b99-93c4-61db55e8ebcc',
 'fd6413c3-0ba6-45ab-9fda-f9045b91db9c',
 '35ff205d-be1d-43e5-843b-bc6ccd0f7a5e',
 '84a7b261-0cda-433c-8449-11de8ec0ad03',
 '8dcb2049-6b08-443a-9ee4-37ffcc06842d',
 '47521071-cd12-4769-9577-67ad2e09f012',
 '594f7a1d-56d9-4301-8a9a-5a55900374a2',
 '76bede06-f5e2-4dc1-8c54-d943bc74e374',
 '9fa917d0-0fae-439d-a467-75943a3fa757',
 '47ce5b23-03a9-4064-b05e-9c1c423cad0f',
 '1eff0fc7-e483-4e77-91ab-af6da273f1bd',
 '6dfacf6b-6382-49e3-bcc2-e7da2acb674b',
 'ecf9bf60-3b3d-4677-be7f-3517ea286520',
 '09e9f109-57a7-

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_template("""
You are an useful assistant.
Answer the following user query with the help of facts provided below.
Query: {query}
here are some facts that might help to answer the query.
{context}

"""

)

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [ ]:
rag_chain = (
    {
        "context": retriever |format_docs,
        "query": RunnablePassthrough(),
    }
    | chat_template
    | llm
    | StrOutputParser()
)



In [ ]:
rag_chain.invoke("which countries were the most represented in the olympics 2024?")

' The most represented countries in the 2024 Summer Olympics were China, the United States, and Russia, with 318, 292, and 272 athletes respectively.'